In [ ]:
import pandas as pd

df = pd.read_csv('starter/data/reviews.csv')  # ← your correct path
df.head()

In [ ]:
# Step 2: Understand Columns

print("Columns:")
print(df.columns)

print("\nTarget Column Value Counts:")
print(df['Recommended IND'].value_counts())

In [9]:
text_feature = 'Review Text'

numerical_features = ['Age', 'Positive Feedback Count']

categorical_features = ['Division Name', 'Department Name', 'Class Name']

target = 'Recommended IND'

In [10]:
# Step 3: Train-Test Split

from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=[target])
y = df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (14753, 8)
Test shape: (3689, 8)


In [11]:
# Handle missing text (IMPORTANT)
X_train[text_feature] = X_train[text_feature].fillna("")
X_test[text_feature] = X_test[text_feature].fillna("")

# Optional: quick check
print("Missing after handling:")
print(X_train.isnull().sum())

Missing after handling:
Clothing ID                0
Age                        0
Title                      0
Review Text                0
Positive Feedback Count    0
Division Name              0
Department Name            0
Class Name                 0
dtype: int64


In [12]:
# Step 4: Build Pipeline

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Text pipeline
text_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=5000))
])

# Numerical pipeline
num_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

# Categorical pipeline
cat_pipeline = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine all
preprocessor = ColumnTransformer([
    ('text', text_pipeline, text_feature),
    ('num', num_pipeline, numerical_features),
    ('cat', cat_pipeline, categorical_features)
])

# Full pipeline
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

In [13]:
# Train model

model_pipeline.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [14]:
# Step 5: Evaluation

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Predictions
y_pred = model_pipeline.predict(X_test)

# Metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

# Full report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.885605855245324
Precision: 0.8984302862419206
Recall: 0.9694453669877118
F1 Score: 0.9325878594249202

Classification Report:

              precision    recall  f1-score   support

           0       0.79      0.51      0.62       678
           1       0.90      0.97      0.93      3011

    accuracy                           0.89      3689
   macro avg       0.84      0.74      0.78      3689
weighted avg       0.88      0.89      0.88      3689


Confusion Matrix:

[[ 348  330]
 [  92 2919]]


In [ ]:
# Step 6: Hyperparameter Tuning

from sklearn.model_selection import GridSearchCV

param_grid = {
    # Text tuning
    'preprocessor__text__tfidf__max_features': [3000, 5000],
    'preprocessor__text__tfidf__ngram_range': [(1,1), (1,2)],
    
    # Model tuning
    'classifier__C': [0.1, 1, 10]
}

grid_search = GridSearchCV(
    model_pipeline,
    param_grid,
    cv=3,
    scoring='f1',
    verbose=2,
    n_jobs=-1
)

# Fit on training data
grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

Fitting 3 folds for each of 12 candidates, totalling 36 fits


In [ ]:
# Best model
best_model = grid_search.best_estimator_

# Predict again
y_pred_best = best_model.predict(X_test)

from sklearn.metrics import classification_report

print("Tuned Model Performance:\n")
print(classification_report(y_test, y_pred_best))